<a href="https://colab.research.google.com/github/avantika-human/painting-artist-detector/blob/main/artworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'best-artworks-of-all-time' dataset.
Path to dataset files: /kaggle/input/best-artworks-of-all-time


In [ ]:
import os
import pandas as pd
file_path = "/kaggle/input/best-artworks-of-all-time"

os.listdir(f"{file_path}/images/images")
datacsv = pd.read_csv(os.path.join(file_path, "artists.csv"))
d1 = datacsv[["name","paintings"]]
print(d1)

                         name  paintings
0           Amedeo Modigliani        193
1          Vasiliy Kandinskiy         88
2                Diego Rivera         70
3                Claude Monet         73
4               Rene Magritte        194
5               Salvador Dali        139
6               Edouard Manet         90
7               Andrei Rublev         99
8            Vincent van Gogh        877
9                Gustav Klimt        117
10           Hieronymus Bosch        137
11           Kazimir Malevich        126
12             Mikhail Vrubel        171
13              Pablo Picasso        439
14          Peter Paul Rubens        141
15      Pierre-Auguste Renoir        336
16             Francisco Goya        291
17                Frida Kahlo        120
18                   El Greco         87
19             Albrecht Dürer        328
20              Alfred Sisley        259
21             Pieter Bruegel        134
22               Marc Chagall        239
23          Giot

In [ ]:
selected_artists = [
    'Claude_Monet',
    'Salvador_Dali',
    'Rembrandt',
    'Vincent_van_Gogh',
    'Leonardo_da_Vinci',
    'Frida_Kahlo',
    'Pablo_Picasso',
    'Michelangelo',
    'Sandro_Botticelli',
    'Pierre-Auguste_Renoir'
]


imagesList = []
artistsList = []

image_path = os.listdir(f"{file_path}/images/images")

for artists in image_path:
  if artists in selected_artists:
    artists_name = artists.replace("_"," ")
    images_path = os.listdir(os.path.join(file_path,"images", "images",artists))
    uptofifty = images_path[:49]

    for images in uptofifty:
      imagesList.append(os.path.join(file_path, "images", "images", artists, images))
      artistsList.append(artists_name)


print(imagesList)

['/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_93.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_15.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_103.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_106.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_59.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_159.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_21.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_31.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_112.jpg', '/kaggle/input/best-artworks-of-all-time/images/images/Sandro_Botticelli/Sandro_Botticelli_146.

In [ ]:
data = pd.DataFrame({"Artists" : artistsList, "Images" : imagesList})
data["Label"] = pd.Categorical(data["Artists"]).codes
data.head()

,Artists,Images,Label
0,Sandro Botticelli,/kaggle/input/best-artworks-of-all-time/images...,8
1,Sandro Botticelli,/kaggle/input/best-artworks-of-all-time/images...,8
2,Sandro Botticelli,/kaggle/input/best-artworks-of-all-time/images...,8
3,Sandro Botticelli,/kaggle/input/best-artworks-of-all-time/images...,8
4,Sandro Botticelli,/kaggle/input/best-artworks-of-all-time/images...,8


In [ ]:

dict(enumerate(pd.Categorical(data["Artists"]).categories))


{0: 'Claude Monet',
 1: 'Frida Kahlo',
 2: 'Leonardo da Vinci',
 3: 'Michelangelo',
 4: 'Pablo Picasso',
 5: 'Pierre-Auguste Renoir',
 6: 'Rembrandt',
 7: 'Salvador Dali',
 8: 'Sandro Botticelli',
 9: 'Vincent van Gogh'}

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
transform = transforms.Compose([
    transforms.Resize((244,244)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class PaintingsDataset(Dataset):
  def __init__(self, dataframe, transform):
    self.dataframe = dataframe
    self.transform = transform

  def __len__(self):
    return len(self.dataframe)

  def __getitem__(self,i):
    row = self.dataframe.iloc[i]
    path = row["Images"]
    label = torch.tensor(row["Label"], dtype = torch.long)
    image = Image.open(path).convert("RGB")
    tensor = self.transform(image)
    return tensor,label



In [ ]:
train_df , test_df = train_test_split(
    data,
    test_size = 0.2,
    stratify = data["Label"],
    random_state = 42
)

In [ ]:
train_ds = PaintingsDataset(train_df, train_transform)
test_ds = PaintingsDataset(test_df, transform)

In [ ]:
train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = False, num_workers=0)


In [26]:
from torchvision import models
model = models.resnet18(pretrained = True)
import torch.nn as nn
model.fc = nn.Linear(512,10)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [27]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [28]:
criterion = nn.CrossEntropyLoss()
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr = 0.0001)


In [29]:
epochs = 20

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for batch in train_loader:
    images, labels = batch
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()


  print(f"Epoch {epoch+1}/{epochs} — Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/20 — Loss: 1.9700
Epoch 2/20 — Loss: 0.8850
Epoch 3/20 — Loss: 0.5413
Epoch 4/20 — Loss: 0.3456
Epoch 5/20 — Loss: 0.2056
Epoch 6/20 — Loss: 0.1491
Epoch 7/20 — Loss: 0.0972
Epoch 8/20 — Loss: 0.0901
Epoch 9/20 — Loss: 0.0749
Epoch 10/20 — Loss: 0.0500
Epoch 11/20 — Loss: 0.0499
Epoch 12/20 — Loss: 0.0364
Epoch 13/20 — Loss: 0.0395
Epoch 14/20 — Loss: 0.0277
Epoch 15/20 — Loss: 0.0314
Epoch 16/20 — Loss: 0.0278
Epoch 17/20 — Loss: 0.0272
Epoch 18/20 — Loss: 0.0243
Epoch 19/20 — Loss: 0.0357
Epoch 20/20 — Loss: 0.0238


Epoch 1/20 — Loss: 1.9700
Epoch 2/20 — Loss: 0.8850
Epoch 3/20 — Loss: 0.5413
Epoch 4/20 — Loss: 0.3456
Epoch 5/20 — Loss: 0.2056
Epoch 6/20 — Loss: 0.1491
Epoch 7/20 — Loss: 0.0972
Epoch 8/20 — Loss: 0.0901
Epoch 9/20 — Loss: 0.0749
Epoch 10/20 — Loss: 0.0500
Epoch 11/20 — Loss: 0.0499
Epoch 12/20 — Loss: 0.0364
Epoch 13/20 — Loss: 0.0395
Epoch 14/20 — Loss: 0.0277
Epoch 15/20 — Loss: 0.0314
Epoch 16/20 — Loss: 0.0278
Epoch 17/20 — Loss: 0.0272
Epoch 18/20 — Loss: 0.0243
Epoch 19/20 — Loss: 0.0357
Epoch 20/20 — Loss: 0.0238



In [30]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 83.67%


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving model4.jpg to model4.jpg


In [ ]:
image_path = "model4.jpg"
pic = Image.open(image_path).convert("RGB")
tensor = transform(pic).unsqueeze(0).to(device)
ans = model(tensor)
print(ans)

tensor([[-1.4985,  1.7119,  0.3200,  1.9207, -1.4027, -0.2121, -1.1111, -1.0917,
         -0.4717, -2.1995]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [31]:
torch.save(model.state_dict(), "painter_model.pth")

In [32]:
from google.colab import files
files.download("painter_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{0: 'Claude Monet',
 1: 'Frida Kahlo',
 2: 'Leonardo da Vinci',
 3: 'Michelangelo',
 4: 'Pablo Picasso',
 5: 'Pierre-Auguste Renoir',
 6: 'Rembrandt',
 7: 'Salvador Dali',
 8: 'Sandro Botticelli',
 9: 'Vincent van Gogh'}



In [ ]:
!pip install gradio -q

In [33]:
import gradio as gr
import torch
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn

# Label map
LABEL_MAP = {
    0: 'Claude Monet',
    1: 'Frida Kahlo',
    2: 'Leonardo da Vinci',
    3: 'Michelangelo',
    4: 'Pablo Picasso',
    5: 'Pierre-Auguste Renoir',
    6: 'Rembrandt',
    7: 'Salvador Dali',
    8: 'Sandro Botticelli',
    9: 'Vincent van Gogh'
}

# Load model
model = models.resnet18()
model.fc = nn.Linear(512, 10)
model.load_state_dict(torch.load("painter_model.pth", map_location="cpu"))
model.eval()

# Transform (same as your val transform)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Prediction function
def predict(image):
    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
    return {LABEL_MAP[i]: float(probs[i]) for i in range(10)}

# Gradio UI
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload a Painting"),
    outputs=gr.Label(num_top_classes=3, label="Artist Prediction"),
    title="🎨 Who Painted This?",
    description="Upload a painting and the model will guess the artist. Trained on 10 iconic painters using ResNet18."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ffab9c6d7fbe32fd03.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [34]:
!git config --global user.email "avantika2607@gmail.com"

In [35]:
!git config --global user.name "avantika-human"

In [36]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [37]:
!git remote add origin https://github.com/avantika-human/painting-artist-detector.git

In [38]:
!git add artworks.ipynb

fatal: pathspec 'artworks.ipynb' did not match any files


In [39]:
!ls

model4.jpg  painter_model.pth  sample_data


In [40]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful